# Notebook 02 - Adapt Dataset With Adaption

This notebook follows the Adaption SDK flow from the docs: install/import the SDK, load credentials, upload the prepared file, confirm column mapping, run an estimate, run a small pilot, poll for completion, download the adapted output, and inspect before/after rows.

**Files read:**
- [`../configs/adaption_run.yaml`](../configs/adaption_run.yaml) - paths, column mapping, pilot size, and timeout settings.
- [`../configs/adaption_blueprint.yaml`](../configs/adaption_blueprint.yaml) - qualitative instructions and Adaption recipe settings.
- [`../data/processed/kakugo_adaption_input.csv`](../data/processed/kakugo_adaption_input.csv) - prepared prompt/response rows from Notebook 01.

**Files written:**
- [`../data/adapted/kakugo_adapted.csv`](../data/adapted/kakugo_adapted.csv) - adapted dataset downloaded from Adaption.
- [`../data/processed/kakugo_adapted_sft.jsonl`](../data/processed/kakugo_adapted_sft.jsonl) - adapted data converted to chat-format JSONL for SFT.
- [`../results/adaption_run_metadata.json`](../results/adaption_run_metadata.json) - run metadata, including the uploaded dataset ID and run plan.
- [`../results/adaption_dataset_diagnosis.json`](../results/adaption_dataset_diagnosis.json) - API-visible dataset description, status, and evaluation metadata captured before the pilot run.- [`../results/adaption_dataset_evaluation.json`](../results/adaption_dataset_evaluation.json) - API-visible quality/evaluation metadata captured after the pilot run.


In [1]:
import json
import os
import sys
from pathlib import Path
import pandas as pd

from adaption import Adaption

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
ROOT_STR = str(ROOT)
if ROOT_STR not in sys.path:
    sys.path.insert(0, ROOT_STR)

In [3]:
from kirundi_sft_starter.data import convert_adapted_to_sft
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml
from scripts.adapt_with_adaption import (
    blueprint_text,
    capture_dataset_diagnosis,
    download_to_file,
    wait_for_completion,
    to_plain_data,
    wait_until_ingested,
)

load_env()

In [4]:
project_config = load_yaml(ROOT / 'configs/project.yaml')
adaption_run_config = load_yaml(ROOT / 'configs/adaption_run.yaml')

blueprint_config = load_yaml(ROOT / 'configs/adaption_blueprint.yaml')

if not os.environ.get('ADAPTION_API_KEY'):
    raise RuntimeError('Missing ADAPTION_API_KEY. Add it to .env before running this notebook.')

## Inspect the input and column mapping

Adaption needs to know which columns contain the prompt and completion. This repo maps `instruction` to `prompt` and `response` to `completion`.

In [5]:
adaption_input_path = ROOT / adaption_run_config['input_path']

adaption_input_df = pd.read_csv(adaption_input_path)

adaption_input_df.head()

,example_id,instruction,response
0,kakugo_00000,Amahitamo aboneka:\n* bibi\n* vyiza\nIncamake ...,bibi
1,kakugo_00001,Kuri igiciro ki igitabo gikoresheje Rs. 150 ki...,"Ukwambere, reka tubare igiciro co kugurisha ki..."
2,kakugo_00002,Ndashaka kumenya uburyo nabona ibikoresho bira...,"Ndagutahura umwuga w’ugura ibikoresho biramba,..."
3,kakugo_00003,Ndakenera kumenya neza ahantu hiyobora kwandik...,Ndagushigikira ko ntashobora gutanga ivyo vyan...
4,kakugo_00004,Hari uburyo bwo kubona cookie runaka hakoreshe...,"Yego! Muri Go, ushobora gukoresha imikorere ya..."


In [6]:
adaption_run_config['column_mapping']

{'prompt': 'instruction', 'completion': 'response'}

In [7]:
print(f"Path: ROOT/{'configs/adaption_blueprint.yaml'}\n")
      
brand_controls = dict(blueprint_config['adaption_brand_controls'])

brand_controls['blueprint'] = blueprint_text(blueprint_config)

print(brand_controls['blueprint'][:1000])

Path: ROOT/configs/adaption_blueprint.yaml

Goal:
Improve this dataset for supervised fine-tuning a small assistant that can answer beginner-friendly questions in Kirundi. The input dataset may contain Kinyarwanda-like or mixed Kinyarwanda/Kirundi examples, so part of the task is language normalization into Kirundi.

Language policy:
- Target language: Kirundi
- Source language issue: Some rows may be mislabeled, mixed, or written in Kinyarwanda-like language rather than clean Kirundi.
- Treat Kirundi as the target language for both the instruction and response.
- If a row is Kinyarwanda-like or mixed but the meaning is clear, rewrite it into natural Kirundi while preserving the original task and answer.
- If the task explicitly asks for another language, preserve that requested output language.
- If local entities, places, currencies, or institutions appear because the row is Kinyarwanda-specific, do not invent Burundi-specific replacements. Prefer neutral wording unless those details

## Upload the dataset

This cell uploads the prepared CSV to Adaption and waits until the platform has ingested enough metadata to report a row count. This is the point where the UI may show its pre-run data diagnosis screen.

In [8]:
client = Adaption(api_key=os.environ['ADAPTION_API_KEY'])
upload = client.datasets.upload_file(str(adaption_input_path), name=adaption_run_config['dataset_name'])
dataset_id = upload.dataset_id
print('Dataset ID:', dataset_id)

wait_until_ingested(
    client,
    dataset_id,
    timeout_seconds=adaption_run_config.get('ingestion_timeout_seconds'),
)
print('Dataset ingestion metadata is available.')

Dataset ID: 70960226-9e25-416f-88bc-537be5036a07
Dataset ingestion metadata is available.


## Run estimate

This cell asks Adaption for a pilot estimate before launching the actual adaptation run. The estimate should not modify the dataset.

In [10]:
estimate = client.datasets.run(
    dataset_id,
    column_mapping=adaption_run_config['column_mapping'],
    brand_controls=brand_controls,
    recipe_specification=blueprint_config['adaption_recipe_specification'],
    job_specification={'max_rows': adaption_run_config['pilot']['max_rows']},
    estimate=True,
)
print('Estimated credits:', estimate.estimated_credits_consumed)

Estimated credits: 1.0


## Run pilot and save outputs

This starts the pilot run, waits for completion, downloads the adapted CSV, and writes run metadata locally. In `configs/adaption_run.yaml`, `timeout_seconds: null` means keep waiting instead of failing after a fixed time. The wait cell prints a heartbeat every 60 seconds.

In [11]:
pilot = client.datasets.run(
    dataset_id,
    column_mapping=adaption_run_config['column_mapping'],
    brand_controls=brand_controls,
    recipe_specification=blueprint_config['adaption_recipe_specification'],
    job_specification={'max_rows': adaption_run_config['pilot']['max_rows']},
)
print('Pilot run started:', pilot.run_id)

Pilot run started: dataset-70960226-9e25-416f-88bc-537be5036a07-1777842475049


In [12]:
final = wait_for_completion(
    client,
    dataset_id,
    timeout_seconds=adaption_run_config['pilot']['timeout_seconds'],
    heartbeat_seconds=60,
)
print('Pilot finished:', final.status)
if getattr(final, 'error', None):
    raise RuntimeError(final.error.message)

Waiting for Adaption run to finish...
Still waiting for Adaption run... elapsed 1m 1s
Still waiting for Adaption run... elapsed 2m 2s
Still waiting for Adaption run... elapsed 3m 3s
Still waiting for Adaption run... elapsed 4m 5s
Still waiting for Adaption run... elapsed 5m 6s
Still waiting for Adaption run... elapsed 6m 7s
Still waiting for Adaption run... elapsed 7m 9s
Still waiting for Adaption run... elapsed 8m 10s
Pilot finished: succeeded


## Retrieve Adaption dataset and evaluation metadata


In [21]:
evaluation_path = ROOT / adaption_run_config.get(
    'evaluation_path',
    'results/adaption_dataset_evaluation.json',
)
ensure_dir(evaluation_path)

post_pilot_evaluation = client.datasets.get_evaluation(dataset_id)
post_pilot_evaluation_data = to_plain_data(post_pilot_evaluation)
evaluation_path.write_text(
    json.dumps(post_pilot_evaluation_data, indent=2, ensure_ascii=False, default=str),
    encoding='utf-8',
)

def find_nested_value(value, key):
    if isinstance(value, dict):
        if key in value:
            return value[key]
        for item in value.values():
            found = find_nested_value(item, key)
            if found is not None:
                return found
    elif isinstance(value, list):
        for item in value:
            found = find_nested_value(item, key)
            if found is not None:
                return found
    return None

In [22]:
evaluation_summary = {
    'grade_before': find_nested_value(post_pilot_evaluation_data, 'grade_before'),
    'grade_after': find_nested_value(post_pilot_evaluation_data, 'grade_after'),
    'score_before': find_nested_value(post_pilot_evaluation_data, 'score_before'),
    'score_after': find_nested_value(post_pilot_evaluation_data, 'score_after'),
    'status': find_nested_value(post_pilot_evaluation_data, 'status'),
    'saved_to': str(evaluation_path.relative_to(ROOT)),
}
pd.DataFrame([evaluation_summary]).T.rename(columns={0: 'value'})


,value
grade_before,C
grade_after,B
score_before,7.0
score_after,8.1
status,pending
saved_to,results/adaption_dataset_evaluation.json


In [13]:
adapted_output_path = ROOT / adaption_run_config['output_path']
metadata_path = ROOT / adaption_run_config['metadata_path']

download_to_file(client, dataset_id, adapted_output_path)

ensure_dir(metadata_path)
metadata = {
    'dataset_id': dataset_id,
    'pilot_run_id': pilot.run_id,
    'input_path': str(adaption_input_path.relative_to(ROOT)),
    'output_path': str(adapted_output_path.relative_to(ROOT)),
    'diagnosis_path': str(diagnosis_path.relative_to(ROOT)),
    'evaluation_path': str(evaluation_path.relative_to(ROOT)) if 'evaluation_path' in globals() else None,
    'column_mapping': adaption_run_config['column_mapping'],
    'pilot': adaption_run_config['pilot'],
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Adapted CSV:', adapted_output_path)
print('Run metadata:', metadata_path)

Adapted CSV: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/adapted/kakugo_adapted.csv
Run metadata: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/adaption_run_metadata.json


In [14]:
adapted_df = pd.read_csv(adapted_output_path)
print('Adapted rows:', len(adapted_df))
adapted_df.head()

Adapted rows: 25


,instruction,response,enhanced_prompt,enhanced_completion,example_id
0,Shiriramo kode igufasha gukora ubu bukora: Yin...,Uno ni porogaramu y'iPython yoroshye igera kur...,Andika kode igufasha gukora ubu bukora:\n\n1. ...,Dore kode yo muri Python igufasha gukora ivyo ...,kakugo_00091
1,Ndashaka igitekerezo cy'ikiganiro giciriritse ...,**Umugabo:** “Ivya birashobora kutwumvirwa nez...,Andika ikiganiro giciriritse hagati y'umugabo ...,"**Umugabo:** ""Maze imyaka icumi ndagutekereza,...",kakugo_00140
2,Ndipfuye kumenya ibiranga inyuguti 'N' na 'G' ...,Inyuguti **N** n’iya **G** ni intungane mu ndi...,Ndipfuye kumenya ibiranga inyuguti 'N' na 'G' ...,"Mu rutonde rw'inyuguti z'Ikirundi, inyuguti 'G...",kakugo_00130
3,Amakuru ava aho mu gihugu ca Kenya amenyesha k...,Abacamanza ba sentare Philemon Mwilu na Isaac ...,### Umwimerere w'Amakuru\nAmakuru ava aho mu g...,1. **Intambwe zafashwe:** Abacamanza Philemon ...,kakugo_00048
4,Mpa ingingo zifatika (byibuze 5) zerekana ubur...,1. **Guhindura inyigisho z’ibiganiro** – Shira...,Tanga ingingo zifatika nibura 5 zerekana ubury...,1. Gushiraho amategeko n'imiyoborere y'uburing...,kakugo_00109


In [15]:
metadata_path = ROOT / adaption_run_config['metadata_path']

with metadata_path.open('r', encoding='utf-8') as f:
    adaption_run_metadata = json.load(f)

pd.DataFrame([adaption_run_metadata]).T.rename(columns={0: 'value'})

,value
dataset_id,70960226-9e25-416f-88bc-537be5036a07
pilot_run_id,dataset-70960226-9e25-416f-88bc-537be5036a07-1...
input_path,data/processed/kakugo_adaption_input.csv
output_path,data/adapted/kakugo_adapted.csv
diagnosis_path,results/adaption_dataset_diagnosis.json
column_mapping,"{'prompt': 'instruction', 'completion': 'respo..."
pilot,"{'max_rows': 25, 'timeout_seconds': None}"


## Convert for SFT

Convert the adapted CSV into the same chat-format JSONL structure used by the raw SFT run.

In [23]:
adapted_sft_path = ROOT / project_config['datasets']['sft']['adapted_sft_path']
adapted_sft_df = convert_adapted_to_sft(adapted_output_path, adapted_sft_path)

print('Converted adapted examples:', len(adapted_sft_df))
print('Adapted SFT JSONL:', adapted_sft_path)

Converted adapted examples: 25
Adapted SFT JSONL: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_adapted_sft.jsonl


In [24]:
with adapted_sft_path.open('r', encoding='utf-8') as f:
    adapted_sft_preview_rows = [json.loads(line) for _, line in zip(range(3), f)]

pd.DataFrame(
    [
        {
            'example_id': row.get('metadata', {}).get('example_id'),
            'user': row['messages'][0]['content'],
            'assistant': row['messages'][1]['content'],
        }
        for row in adapted_sft_preview_rows
    ]
)

,example_id,user,assistant
0,kakugo_00091,Shiriramo kode igufasha gukora ubu bukora: Yin...,Uno ni porogaramu y'iPython yoroshye igera kur...
1,kakugo_00140,Ndashaka igitekerezo cy'ikiganiro giciriritse ...,**Umugabo:** “Ivya birashobora kutwumvirwa nez...
2,kakugo_00130,Ndipfuye kumenya ibiranga inyuguti 'N' na 'G' ...,Inyuguti **N** n’iya **G** ni intungane mu ndi...


## CLI Equivalent

```bash
python scripts/adapt_with_adaption.py
# Optional full run after the pilot looks good:
python scripts/adapt_with_adaption.py --full
```

The notebook conversion step above is equivalent to:

```bash
python scripts/convert_adapted_to_sft.py
```